# Нагрузочное тестирование пайплайна

Гоняет Perf Analyzer по пяти сценариям, снимает загрузку железа,
собирает таблицы и пишет `report/results.md`.

Перед запуском сервер должен быть поднят: `docker compose up -d`.

## 0. Образ с Perf Analyzer

`perf_analyzer` — не сервис, а бинарник внутри образа `…-py3-sdk`. Каждый замер
делает `docker run --rm`: одноразовый контейнер, одна команда.
В `docker ps` его не видно. Долгоживущий контейнер один — `triton` из compose.

Ячейку достаточно выполнить один раз на машине.

In [2]:
import subprocess

SDK_IMAGE = "nvcr.io/nvidia/tritonserver:25.05-py3-sdk"
# тянется анонимно. Если откажет — зеркало:
# SDK_IMAGE = "hubimage/nvcr-io-nvidia-tritonserver:25.05-py3-sdk"

subprocess.run(["docker", "pull", SDK_IMAGE], check = True)
# проверим что скачалось и рабочее
result = subprocess.run(
    ["docker", "run", "--rm", SDK_IMAGE, "perf_analyzer", "--version"],
    capture_output = True, text = True, encoding = "utf-8", check = True,
)
print(result.stdout, result.stderr)


== Triton Inference Server SDK ==

NVIDIA Release 25.05 (build 170551373)

Copyright (c) 2018-2025, NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

Various files include modifications (c) NVIDIA CORPORATION & AFFILIATES.  All rights reserved.

GOVERNING TERMS: The software and materials are governed by the NVIDIA Software License Agreement
(found at https://www.nvidia.com/en-us/agreements/enterprise-software/nvidia-software-license-agreement/)
and the Product-Specific Terms for NVIDIA AI Products
(found at https://www.nvidia.com/en-us/agreements/enterprise-software/product-specific-terms-for-ai-products/).

   Use the NVIDIA Container Toolkit to start this container with GPU support; see
   https://docs.nvidia.com/datacenter/cloud-native/ .

 Perf Analyzer Version 2.58.0 (commit unknown)



## 1. Константы и проверка READY

Ноутбук ходит в Triton с хоста, а
`perf_analyzer` — из соседнего контейнера, для которого хост называется
`host.docker.internal`. 

In [3]:
import json
import re
import sys
import time
import urllib.request
from pathlib import Path

import pandas as pd
import tritonclient.grpc as grpcclient

CLIENT_URL = "localhost:8001"              # ноутбук -> Triton, с хоста
TRITON_URL = "host.docker.internal:8001"   # perf_analyzer из контейнера -> Triton

REPO      = Path("..")                     # ноутбук лежит в notebooks/
BENCH_DIR = REPO / "bench"                 # монтируется в контейнер как /bench
RAW_DIR   = REPO / "report" / "raw"

MODELS = ["toxicity_clf", "e5_embedder", "text_generator", "assistant_bls"]

BENCH_DIR.mkdir(exist_ok = True)
RAW_DIR.mkdir(parents = True, exist_ok = True)


def list_models(url = CLIENT_URL):
    """Печатает статус каждой модели репозитория.

        url: адрес gRPC-порта Triton.

    Возвращает: True, если все модели из MODELS готовы.
    """
    client = grpcclient.InferenceServerClient(url = url)

    lines = []
    for model in client.get_model_repository_index().models:
        ready = client.is_model_ready(model.name)
        lines.append(f"   {'✅ ready' if ready else '❌ not ready'}  {model.name}")

    report = "Модели в репозитории:\n" + "\n".join(lines)
    print(report)
    # этот же вывод пойдёт в отчёт вместо скриншота
    (RAW_DIR / "models_ready.txt").write_text(report, encoding = "utf-8")

    return all(client.is_model_ready(name) for name in MODELS)


assert list_models(), "не все модели READY — смотрите docker compose logs triton"

Модели в репозитории:
   ✅ ready  assistant_bls
   ✅ ready  e5_embedder
   ✅ ready  text_generator
   ✅ ready  toxicity_clf


## 2. Файлы входных данных

`bench/*.json` для моделей toxicity и e5 пишет эта ячейка — из словарей
и корпуса, которые уже уехали в `assistant_bls/1/`. 

In [20]:
from transformers import AutoTokenizer

BLS = REPO / "model_repository" / "assistant_bls" / "1"
sys.path.append(str(BLS))              # иначе ModuleNotFoundError на bls_prompt
from bls_prompt import build_messages

MAX_LENGTH = 128                       # столько же, сколько в model.py BLS

QUESTIONS = [
    "где в Кисловодске покататься на канатной дороге",
    "куда сходить с детьми в Железноводске",
    "какие источники есть в Ессентуках",
]
RUDE = [
    "ты тупой бот, отвечай нормально",
    # без "идиоты" классификатор не считает фразу токсичной
    "работаете отвратительно, идиоты, сколько можно ждать",
]


def dump(name, entries):
    """Пишет файл в формате Perf Analyzer.

        name: имя файла в каталоге bench.
        entries: список записей, по одной на запрос.

    Возвращает: Ничего.
    """
    path = BENCH_DIR / name
    path.write_text(json.dumps({"data": entries}, ensure_ascii = False), encoding = "utf-8")
    print(f"{path}  записей: {len(entries)}")


def token_entries(tokenizer_dir, texts, keys):
    """Токенизирует тексты и раскладывает по входам модели.

        tokenizer_dir: папка со словарём.
        texts: строки для токенизации.
        keys: имена входных тензоров модели.

    Возвращает: Список записей для Perf Analyzer.
    """
    tokenizer = AutoTokenizer.from_pretrained(str(tokenizer_dir))
    encoded = tokenizer(texts, padding = "max_length", truncation = True, max_length = MAX_LENGTH)

    entries = []
    for index in range(len(texts)):
        entry = {}
        for key in keys:
            # у e5 token_type_ids нет — BLS в этом месте подставляет нули, и мы тоже
            values = encoded[key][index] if key in encoded else [0] * MAX_LENGTH
            entry[key] = {"content": [int(v) for v in values], "shape": [MAX_LENGTH]}
        entries.append(entry)

    return entries


dump("tox.json", token_entries(
    BLS / "tokenizers" / "ru", QUESTIONS + RUDE,
    ["input_ids", "attention_mask", "token_type_ids"],
))

dump("e5.json", token_entries(
    BLS / "tokenizers" / "e5", [f"query: {q}" for q in QUESTIONS],
    ["input_ids", "attention_mask"],
))

..\bench\tox.json  записей: 5
..\bench\e5.json  записей: 3


Генератор. 

In [21]:

# три самых длинных документа
documents = sorted(
    json.loads((BLS / "corpus" / "documents.json").read_text(encoding = "utf-8")),
    key = lambda document: -len(document["context_text"]),
)[:3]
qwen = AutoTokenizer.from_pretrained(str(BLS / "tokenizers" / "qwen"))
SAMPLING = {"temperature": 0.7, "top_p": 0.8, "top_k": 20, "min_p": 0.0, "max_tokens": 256}

gen_entries = []
for question in QUESTIONS:
    prompt = qwen.apply_chat_template(
        build_messages(question, documents),
        tokenize = False, add_generation_prompt = True, enable_thinking = False,
    )
    gen_entries.append({
        "text_input": [prompt],
        "stream": [False],
        "exclude_input_in_output": [True],
        "sampling_parameters": [json.dumps(SAMPLING)],
    })

dump("gen.json", gen_entries)
print("токенов в последнем промпте:", len(qwen(prompt)["input_ids"]))

dump("bls_questions.json", [{"TEXT": [q]} for q in QUESTIONS])
dump("bls_rude.json", [{"TEXT": [r]} for r in RUDE])

..\bench\gen.json  записей: 3
токенов в последнем промпте: 1287
..\bench\bls_questions.json  записей: 3
..\bench\bls_rude.json  записей: 2


## 3. Функция запуска

Каждый вызов поднимает одноразовый sdk-контейнер, монтирует в него `bench/`
и забирает CSV. 

In [22]:
all_frames = []                # сюда run_perf складывает каждый прогон

SLOW = (
    "--measurement-mode", "count_windows",
    "--measurement-request-count", "20",
    "--stability-percentage", "35",
    "--warmup-request-count", "5",
)


def run_perf(model, data_file, concurrency, tag, extra = ()):
    """Гоняет perf_analyzer в контейнере sdk.

        model: имя модели в репозитории.
        data_file: имя файла из bench, внутри контейнера он лежит в /bench.
        concurrency: сколько одновременных запросов держать.
        tag: метка прогона, попадает в имена файлов.
        extra: дополнительные ключи perf_analyzer.

    Возвращает: DataFrame из CSV, который написал perf_analyzer.
    """
    csv_name = f"{tag}_c{concurrency}.csv"

    command = [
        "docker", "run", "--rm",
        # as_posix обязателен: resolve() под Windows даёт C:\...\bench,
        # docker такой путь в -v не разбирает
        "-v", f"{BENCH_DIR.resolve().as_posix()}:/bench",
        "-v", f"{RAW_DIR.resolve().as_posix()}:/raw",

        SDK_IMAGE, "perf_analyzer",
        "-m", model,
        "-u", TRITON_URL,
        "-i", "grpc",
        "--input-data", f"/bench/{data_file}",
        "--concurrency-range", f"{concurrency}:{concurrency}",
        "-f", f"/raw/{csv_name}",
        *extra,
    ]

    print(" ".join(command))
    # без timeout ячейка висит молча, если прогон не сходится
    result = subprocess.run(
        command, capture_output = True, text = True, encoding = "utf-8", timeout = 1800,
    )

    (RAW_DIR / f"{tag}_c{concurrency}.txt").write_text(
        (result.stdout or "") + (result.stderr or ""), encoding = "utf-8",
    )

    if result.returncode != 0:
        # perf_analyzer при ошибке аргументов печатает всё полотно usage,
        # настоящая причина — в самом конце вывода
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])
        raise RuntimeError(f"perf_analyzer упал на {model}, concurrency {concurrency}")

    frame = pd.read_csv(RAW_DIR / csv_name)
    frame.insert(0, "scenario", tag)
    frame.insert(1, "model", model)

    all_frames.append(frame)   # копится для сводки в ячейке 8

    return frame

Проверка на одной модели, прежде чем гонять всё. Если тут ошибка — читайте
последнюю строку `report/raw/smoke_c1.txt`.

In [ ]:
run_perf("toxicity_clf", "tox.json", 1, "smoke")
all_frames.clear()

docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m toxicity_clf -u host.docker.internal:8001 -i grpc --input-data /bench/tox.json --concurrency-range 1:1 -f /raw/smoke_c1.csv


## 4. Прогон отдельных моделей

Классификатор и эмбеддер — обычные ONNX-модели с `dynamic_batching`, им
подходят уровни 1 / 8 / 16.

In [12]:
STABLE = ("--measurement-interval", "10000", "--stability-percentage", "20")

for concurrency in [1, 8, 16]:
    run_perf("toxicity_clf", "tox.json", concurrency, "tox", extra = STABLE)

for concurrency in [1, 8, 16]:
    run_perf("e5_embedder", "e5.json", concurrency, "e5", extra = STABLE)

docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m toxicity_clf -u host.docker.internal:8001 -i grpc --input-data /bench/tox.json --concurrency-range 1:1 -f /raw/tox_c1.csv --measurement-interval 10000 --stability-percentage 20
docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m toxicity_clf -u host.docker.internal:8001 -i grpc --input-data /bench/tox.json --concurrency-range 8:8 -f /raw/tox_c8.csv --measurement-interval 10000 --stability-percentage 20
docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m toxicity_clf -u host.docker.internal:8001 -i grpc --input-data /bench/

e5_embedder с разными параметрами распаралеривания:

In [ ]:
# instance_group [ { count: 1, kind: KIND_CPU } ]
#run_perf("e5_embedder", "e5.json", 16, "e5_count1")   # текущий конфиг



docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m e5_embedder -u host.docker.internal:8001 -i grpc --input-data /bench/e5.json --concurrency-range 16:16 -f /raw/e5_count1_c16.csv


,scenario,model,Concurrency,Inferences/Second,Client Send,Network+Server Send/Recv,Server Queue,Server Compute Input,Server Compute Infer,Server Compute Output,Client Recv,p50 latency,p90 latency,p95 latency,p99 latency
0,e5_count1,e5_embedder,16,28.0827,12,11085,314218,238,241575,78,7,547320,736089,819741,862987


In [ ]:
# правим config.pbtxt на count: 2, restart, ждём READY

# instance_group [ { count: 2, kind: KIND_CPU } ]
# parameters {
#   key: "intra_op_thread_count"
#   value: { string_value: "2" }
# }
#run_perf("e5_embedder", "e5.json", 16, "e5_count2")



docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m e5_embedder -u host.docker.internal:8001 -i grpc --input-data /bench/e5.json --concurrency-range 16:16 -f /raw/e5_count2_c16.csv


,scenario,model,Concurrency,Inferences/Second,Client Send,Network+Server Send/Recv,Server Queue,Server Compute Input,Server Compute Infer,Server Compute Output,Client Recv,p50 latency,p90 latency,p95 latency,p99 latency
0,e5_count2,e5_embedder,16,20.4194,12,6946,212950,90,559209,69,6,787959,1124635,1137769,1239394


In [ ]:
# правим на count: 4 x 4
# instance_group [ { count: 4, kind: KIND_CPU } ]
# parameters {
#   key: "intra_op_thread_count"
#   value: { string_value: "4" }
# }
#run_perf("e5_embedder", "e5.json", 16, "e5_count4x4")

docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m e5_embedder -u host.docker.internal:8001 -i grpc --input-data /bench/e5.json --concurrency-range 16:16 -f /raw/e5_count4x4_c16.csv


,scenario,model,Concurrency,Inferences/Second,Client Send,Network+Server Send/Recv,Server Queue,Server Compute Input,Server Compute Infer,Server Compute Output,Client Recv,p50 latency,p90 latency,p95 latency,p99 latency
0,e5_count4x4,e5_embedder,16,34.1564,15,12253,68216,128,382502,88,8,453335,685241,719468,854972


In [17]:
# правим на count: 4 + распаралеривание внутри onnx модели на 2, чтобы занимать 8, а не все 16 ядер
# instance_group [ { count: 4, kind: KIND_CPU } ]
# parameters {
#   key: "intra_op_thread_count"
#   value: { string_value: "2" }
# }
run_perf("e5_embedder", "e5.json", 16, "e5_count4x2")

# эти же числа будут в полном прогоне

docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m e5_embedder -u host.docker.internal:8001 -i grpc --input-data /bench/e5.json --concurrency-range 16:16 -f /raw/e5_count4x2_c16.csv


,scenario,model,Concurrency,Inferences/Second,Client Send,Network+Server Send/Recv,Server Queue,Server Compute Input,Server Compute Infer,Server Compute Output,Client Recv,p50 latency,p90 latency,p95 latency,p99 latency
0,e5_count4x2,e5_embedder,16,26.1611,16,9409,100585,207,493271,93,7,615015,822628,942706,1172542


## 5. Прогон генератора

- **`--streaming`**: бэкенд vLLM в `auto_complete_config` безусловно ставит
  `decoupled: True`. К такой модели Perf Analyzer должен идти
  gRPC-потоком. `--async` требуется вместе со `--streaming`.
- **Уровни 1 / 4 / 8**: на 4 ГБ под KV-кэш остаётся меньше гигабайта,
  шестнадцать запросов туда не влезут, vLLM начнёт вытеснять их друг другом,
  и прогон растянется на минуты.

In [13]:
STREAM = ("--async", "--streaming", *SLOW)

for concurrency in [1, 4, 8]:
    run_perf("text_generator", "gen.json", concurrency, "gen", extra = STREAM)

docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m text_generator -u host.docker.internal:8001 -i grpc --input-data /bench/gen.json --concurrency-range 1:1 -f /raw/gen_c1.csv --async --streaming --measurement-mode count_windows --measurement-request-count 20 --stability-percentage 35
docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m text_generator -u host.docker.internal:8001 -i grpc --input-data /bench/gen.json --concurrency-range 4:4 -f /raw/gen_c4.csv --async --streaming --measurement-mode count_windows --measurement-request-count 20 --stability-percentage 35
docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/

## 6. Прогон цепочки через BLS

Два сценария: обычные вопросы (идут в поиск и генератор) и грубые (отсекаются
классификатором, генератор не вызывается). 

Уровни 1 / 2 / 4: в `assistant_bls/config.pbtxt` стоит `instance_group
{ count: 2 }`, параллельно исполняются ровно два запроса. При concurrency 16
четырнадцать будут стоять в очереди.

In [ ]:
# ветка полной обработки с llm
for concurrency in [1, 2, 4]:
    run_perf("assistant_bls", "bls_questions.json", concurrency, "bls_questions", extra = SLOW)

# ветка с грубым обращением
for concurrency in [1, 2, 4]:
    run_perf("assistant_bls", "bls_rude.json", concurrency, "bls_rude", extra = SLOW)

docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m assistant_bls -u host.docker.internal:8001 -i grpc --input-data /bench/bls_questions.json --concurrency-range 1:1 -f /raw/bls_questions_c1.csv --measurement-mode count_windows --measurement-request-count 20 --stability-percentage 35
docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tritonserver:25.05-py3-sdk perf_analyzer -m assistant_bls -u host.docker.internal:8001 -i grpc --input-data /bench/bls_questions.json --concurrency-range 2:2 -f /raw/bls_questions_c2.csv --measurement-mode count_windows --measurement-request-count 20 --stability-percentage 35
docker run --rm -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench -v C:/devs/AI/LlmEngineer/llm-eng-26-triton/report/raw:/raw nvcr.io/nvidia/tr

## 6б. Загрузка железа

`nvidia-smi` и метрики Triton на порту 8002. 

Снимаем и то, и другое — в покое и под нагрузкой генератора, только он грузит карту.

In [24]:
def snapshot(tag):
    """Пишет nvidia-smi и метрики Triton в report/raw/.

        tag: метка момента съёмки, попадает в имена файлов.

    Возвращает: Ничего.
    """
    smi = subprocess.run(
        ["docker", "exec", "triton", "nvidia-smi"],
        capture_output = True, text = True, encoding = "utf-8",
    ).stdout
    (RAW_DIR / f"nvidia_smi_{tag}.txt").write_text(smi or "", encoding = "utf-8")

    metrics = urllib.request.urlopen("http://localhost:8002/metrics").read().decode("utf-8")
    (RAW_DIR / f"metrics_{tag}.txt").write_text(metrics, encoding = "utf-8")

    print(smi)


snapshot("idle")

Wed Sep  9 10:56:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 591.59         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   48C    P8              1W /   50W |    2403MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Теперь то же самое под нагрузкой: прогон генератора уходит в фон, снимок
делается, пока он идёт.

In [27]:
load = subprocess.Popen(
    [
        "docker", "run", "--rm",
        "-v", f"{BENCH_DIR.resolve().as_posix()}:/bench",
        SDK_IMAGE, "perf_analyzer",
        "-m", "text_generator", "-u", TRITON_URL, "-i", "grpc",
        "--input-data", "/bench/gen.json", "--concurrency-range", "4:4",
        *STREAM,
    ],
    stdout = subprocess.DEVNULL, stderr = subprocess.DEVNULL,
)

time.sleep(30)          # даём нагрузке разойтись
snapshot("gen_load")

# докер больше не нужен - не грузим
subprocess.run(["docker", "kill", "bench_load"], capture_output = True)
load.wait(timeout = 60)


Wed Sep  9 11:08:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 591.59         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce GTX 1650        On  |   00000000:01:00.0 Off |                  N/A |
| N/A   62C    P0             26W /   50W |    2433MiB /   4096MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

TimeoutExpired: Command '['docker', 'run', '--rm', '-v', 'C:/devs/AI/LlmEngineer/llm-eng-26-triton/bench:/bench', 'nvcr.io/nvidia/tritonserver:25.05-py3-sdk', 'perf_analyzer', '-m', 'text_generator', '-u', 'host.docker.internal:8001', '-i', 'grpc', '--input-data', '/bench/gen.json', '--concurrency-range', '4:4', '--async', '--streaming', '--measurement-mode', 'count_windows', '--measurement-request-count', '20', '--stability-percentage', '35']' timed out after 60 seconds

## 7. Батчинг и без него

`dynamic_batching` есть только у двух ONNX-моделей — сравниваем их.
У vLLM свой непрерывный батчинг, у BLS батчинга нет.

Ячейка вырезает блок из конфигов, перезапускает сервер, повторяет прогоны
с тегом `_nobatch` и возвращает конфиги на место. Оригиналы лежат рядом
в `.bak` — если ячейка упадёт посередине, то надо восстановить их руками.

In [ ]:
BATCHED = {
    "toxicity_clf": "tox",
    "e5_embedder": "e5",
}
BLOCK = re.compile(r"dynamic_batching\s*\{[^}]*\}\s*", re.S)


def wait_ready(timeout = 300):
    """Ждёт, пока все модели снова поднимутся.

        timeout: сколько секунд ждать.

    Возвращает: Ничего.
    """
    client = grpcclient.InferenceServerClient(url = CLIENT_URL)
    deadline = time.time() + timeout

    while time.time() < deadline:
        try:
            if all(client.is_model_ready(name) for name in MODELS):
                print("все модели READY")
                return
        except Exception:
            pass                      # сервер ещё поднимается, соединения нет
        time.sleep(5)

    raise TimeoutError("модели не поднялись")


originals = {}
for name in BATCHED:
    config = REPO / "model_repository" / name / "config.pbtxt"
    originals[name] = config.read_text(encoding = "utf-8")
    config.with_suffix(".pbtxt.bak").write_text(originals[name], encoding = "utf-8")
    config.write_text(BLOCK.sub("", originals[name]), encoding = "utf-8")
    print(f"{name}: dynamic_batching вырезан")

subprocess.run(["docker", "compose", "restart", "triton"], cwd = REPO, check = True)
wait_ready()

In [ ]:
try:
    for name, tag in BATCHED.items():
        for concurrency in [1, 8, 16]:
            run_perf(name, f"{tag}.json", concurrency, f"{tag}_nobatch")
finally:
    # конфиги возвращаем в любом случае, даже если прогон упал
    for name, text in originals.items():
        (REPO / "model_repository" / name / "config.pbtxt").write_text(text, encoding = "utf-8")
        (REPO / "model_repository" / name / "config.pbtxt.bak").unlink(missing_ok = True)

    subprocess.run(["docker", "compose", "restart", "triton"], cwd = REPO, check = True)
    wait_ready()

## 8. Сводная таблица и график

Колонки `avg latency` в CSV нет — она складывается из семи слагаемых,
все в микросекундах. Полный список того, что пишет Perf Analyzer:

```
Concurrency, Inferences/Second,
Client Send, Network+Server Send/Recv, Server Queue,
Server Compute Input, Server Compute Infer, Server Compute Output, Client Recv,
p50 latency, p90 latency, p95 latency, p99 latency
```

In [ ]:
LATENCY_PARTS = [
    "Client Send", "Network+Server Send/Recv", "Server Queue",
    "Server Compute Input", "Server Compute Infer", "Server Compute Output",
    "Client Recv",
]

summary = pd.concat(all_frames, ignore_index = True)   # заполнен в ячейке 3
summary["avg latency"] = summary[LATENCY_PARTS].sum(axis = 1)

# в отчёт — миллисекунды, микросекунды никто не читает
for column in LATENCY_PARTS + ["avg latency", "p95 latency"]:
    summary[column] = summary[column] / 1000

table = summary[[
    "scenario", "model", "Concurrency", "Inferences/Second",
    "avg latency", "p95 latency", "Server Queue", "Server Compute Infer",
]].round(2).sort_values(["scenario", "Concurrency"])

table.to_csv(REPO / "report" / "summary.csv", index = False)
table

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize = (13, 5))

for scenario, group in table.groupby("scenario"):
    group = group.sort_values("Concurrency")
    axes[0].plot(group["Concurrency"], group["Inferences/Second"], marker = "o", label = scenario)
    axes[1].plot(group["Concurrency"], group["avg latency"], marker = "o", label = scenario)

# генератор на два порядка медленнее остальных: в линейной шкале
# всё остальное схлопнется в ноль
for axis, title, ylabel in [
    (axes[0], "Пропускная способность", "запросов в секунду"),
    (axes[1], "Средняя задержка", "мс"),
]:
    axis.set_yscale("log")
    axis.set_xlabel("concurrency")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(True, which = "both", alpha = 0.3)
    axis.legend(fontsize = 8)

figure.tight_layout()
figure.savefig(REPO / "report" / "bench.png", dpi = 120)
plt.show()

## 9. Сборка отчёта

Пишет `report/results.md`: дерево репозитория, конфиги, статус READY, таблицы,
график, ссылки на сырые логи. Разделы с выводами оставлены заготовками —
их дописывать словами, глядя на свои же цифры.

In [ ]:
def tree(root, prefix = ""):
    """Рисует дерево каталога без внешних утилит.

        root: корень обхода.
        prefix: служебный отступ для рекурсии.

    Возвращает: Список строк.
    """
    entries = sorted(root.iterdir(), key = lambda path: (path.is_file(), path.name))
    lines = []

    for index, entry in enumerate(entries):
        last = index == len(entries) - 1
        lines.append(f"{prefix}{'└── ' if last else '├── '}{entry.name}")
        if entry.is_dir():
            lines += tree(entry, prefix + ("    " if last else "│   "))

    return lines


model_repository = REPO / "model_repository"
configs = "\n\n".join(
    f"### `{path.parent.name}/config.pbtxt`\n\n```protobuf\n{path.read_text(encoding='utf-8').strip()}\n```"
    for path in sorted(model_repository.glob("*/config.pbtxt"))
)

ready = (RAW_DIR / "models_ready.txt").read_text(encoding = "utf-8")

report = f"""# Отчёт: оркестрация моделей в Triton

## 1. Дерево Model Repository

```
model_repository
{chr(10).join(tree(model_repository))}
```

## 2. Конфигурация моделей

{configs}

Код BLS — [`model_repository/assistant_bls/1/model.py`](../model_repository/assistant_bls/1/model.py),
сборка промпта — [`bls_prompt.py`](../model_repository/assistant_bls/1/bls_prompt.py).

## 3. Все модели READY

```
{ready}
```

## 4. Замеры

{table.to_markdown(index = False)}

Задержки в миллисекундах, `avg latency` — сумма всех этапов, от отправки
клиентом до получения ответа.

![Пропускная способность и задержка](bench.png)

## 5. Батчинг и без него

Сценарии `*_nobatch` — те же модели с вырезанным блоком `dynamic_batching`.

TODO: сравнить строки `tox` и `tox_nobatch`, `e5` и `e5_nobatch`, написать вывод.

## 6. Загрузка железа

`nvidia-smi` в покое и под нагрузкой генератора:
[`raw/nvidia_smi_idle.txt`](raw/nvidia_smi_idle.txt),
[`raw/nvidia_smi_gen_load.txt`](raw/nvidia_smi_gen_load.txt).
Метрики Triton с порта 8002: [`raw/metrics_gen_load.txt`](raw/metrics_gen_load.txt),
смотреть `nv_inference_request_duration_us` и `nv_inference_queue_duration_us`
по каждой модели.

TODO: вписать занятую видеопамять и утилизацию GPU под нагрузкой.

## 7. Узкое место

TODO: сравнить `bls_questions` и `bls_rude` — во втором генератор не вызывается.
Разница в RPS показывает его долю в стоимости запроса.

## 8. Чем Triton лучше FastAPI

- промежуточные тензоры не сериализуются и не ходят по сети: BLS вызывает
  модели внутри сервера
- батчинг настраивается конфигом, а не пишется руками
- число копий модели меняется строкой `instance_group`, без правки кода
- метрики по каждой модели отдельно есть из коробки, порт 8002

## 9. Предложения по оптимизации

TODO: опереться на свои цифры. Кандидаты: `instance_group` у BLS, квантование
эмбеддера в int8, переменная длина вместо фиксированных 128 токенов, ранний
отсев грубых сообщений до генератора.

## 10. Сырые логи

Все прогоны Perf Analyzer: [`raw/`](raw/), сводка: [`summary.csv`](summary.csv).
"""

(REPO / "report" / "results.md").write_text(report, encoding = "utf-8")
print(report[:1500])